### This notebook will simulate an external application which periodically loads data to landing.raw_files

## Imports and configuration

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

source_path = "/Volumes/retail_project/external_app_schema/external_app_volume/Year_2010-2011.csv"

target_table = (
    "retail_project.external_app_schema.future_transactions"
)

In [0]:
# Define schema

schema = StructType([
    StructField("Invoice", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", TimestampType(), True),
    StructField("Price", DecimalType(12, 4), True),
    StructField("Customer ID", DoubleType(), True),
    StructField("Country", StringType(), True)
])

## Reading data

In [0]:
df = (
    spark.read
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(schema)
    .csv(source_path)
)

df = (
    df
    .withColumnRenamed("Customer ID", "CustomerID")
    .withColumn("transaction_date", to_date(col("InvoiceDate")))
)

## Validation

In [0]:
df.agg(
    count("*").alias("total_rows"),
    countDistinct("transaction_date").alias("distinct_days"),
    min("transaction_date").alias("first_date"),
    max("transaction_date").alias("last_date"),
    sum(
        when(col("transaction_date").isNull(), 1).otherwise(0)
    ).alias("null_dates")
).show()

In [0]:
daily_counts = (
    df.groupBy("transaction_date")
    .agg(count("*").alias("row_count"))
    .orderBy("transaction_date")
)

display(daily_counts)

## Write to delta table

In [0]:
df.write.format("delta").saveAsTable(target_table)